In [1]:
#Here each molecule is considered as a node. So here there is only one graph.

In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
import networkx as nx

In [3]:
import matplotlib.pyplot as plt

In [4]:
df1 = pd.read_csv('./maoa_maob_common.csv', na_values=['na'])
df1 = df1.dropna(ignore_index=True)
df1

,molecule_chembl_id,smiles,maoa_ic50,maob_ic50
0,CHEMBL50,O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12,6665.00,38660.00
1,CHEMBL98,O=C(CCCCCCC(=O)Nc1ccccc1)NO,100000.00,100000.00
2,CHEMBL113,Cn1c(=O)c2c(ncn2C)n(C)c1=O,50000.00,50000.00
3,CHEMBL126,CC(=O)NC[C@H]1CN(c2ccc(N3CCOCC3)c(F)c2)C(=O)O1,46000.00,2100.00
4,CHEMBL140,COc1cc(/C=C/C(=O)CC(=O)/C=C/c2ccc(O)c(OC)c2)ccc1O,3880.00,2560.00
...,...,...,...,...
2952,CHEMBL6069449,Fc1ccc([C@@H]2C[C@H]2NCc2cn(-c3ccc(-c4ccccc4)c...,3.22,371.00
2953,CHEMBL6069644,Fc1ccc([C@@H]2C[C@H]2NCc2cn(-c3ccc(-c4ccncc4)c...,28.61,2310.00
2954,CHEMBL6069736,Fc1ccc([C@@H]2C[C@H]2NCc2cn(-c3ccc(N4CCCC4)cc3...,18.65,402.57
2955,CHEMBL6069771,Fc1ccc([C@@H]2C[C@H]2NCc2cn(-c3ccc4[nH]ncc4c3)...,242.97,8049.67


In [5]:
df2 = df1.loc[:, ['maoa_ic50', 'maob_ic50']]
df2

,maoa_ic50,maob_ic50
0,6665.00,38660.00
1,100000.00,100000.00
2,50000.00,50000.00
3,46000.00,2100.00
4,3880.00,2560.00
...,...,...
2952,3.22,371.00
2953,28.61,2310.00
2954,18.65,402.57
2955,242.97,8049.67


In [6]:
for i in range(len(df2)):
    if df2['maoa_ic50'].values[i] <= 1000:
        df2.loc[i, ['maoa_ic50']] = 1
    else: 
        df2.loc[i, ['maoa_ic50']] = 0

In [7]:
for i in range(len(df2)):
    if df2['maob_ic50'].values[i] <= 1000:
        df2.loc[i, ['maob_ic50']] = 1
    else: 
        df2.loc[i, ['maob_ic50']] = 0

In [8]:
newcol = []
for i in range(len(df2)):
    if df2['maoa_ic50'].values[i] == 1.0 and df2['maob_ic50'].values[i] == 1.0:
        newcol.append('AA')
    elif df2['maoa_ic50'].values[i] == 0.0 and df2['maob_ic50'].values[i] == 0.0:
        newcol.append('II')
    elif df2['maoa_ic50'].values[i] == 0.0 and df2['maob_ic50'].values[i] == 1.0:
        newcol.append('IA')
    else:
        newcol.append('AI')

In [9]:
target = pd.DataFrame(data=newcol, columns=['maoa_maob'])
target

,maoa_maob
0,II
1,II
2,II
3,II
4,II
...,...
2952,AA
2953,AI
2954,AA
2955,AI


In [10]:
y = target.values.reshape(len(df2),)
y.shape

(2957,)

In [11]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [12]:
y_label = encoder.fit_transform(y)

In [13]:
y_label

array([3, 3, 3, ..., 0, 1, 1])

In [14]:
from rdkit.Chem import SaltRemover as sr
remover = sr.SaltRemover()

In [15]:
mols = []
for i in range(len(df1)):
    try:
        mol_i = Chem.MolFromSmiles(df1['smiles'][i])
        if mol_i is None:
            print(f"[WARNING] Invalid SMILES at index {i}, skipping.")
            mols.append(None)  # Maintain indexing for multiprocessing
            continue
        mol_i = remover.StripMol(mol_i, dontRemoveEverything=True)
        mols.append(mol_i)
    except Exception as e:
        print(e)
len(mols)

2957

In [16]:
smiles_list = [df1['smiles'][i] for i in range(len(df1['smiles']))]

In [17]:
names_list = [df1['molecule_chembl_id'][i] for i in range(len(df1['molecule_chembl_id']))]

In [18]:
#Deprecated method!
FPs = [AllChem.GetMorganFingerprintAsBitVect(mols[i], radius=3, nBits=1024) for i in range(len(mols))]

[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerator
[12:57:36] DEPRECATION WARNING: please use MorganGenerat

In [19]:
similarity_matrix = np.zeros((len(FPs), len(FPs)))

In [20]:
similarity_matrix.shape

(2957, 2957)

In [21]:
#only works for bit vectors or sparse count vectors, not arbitrary numerical vectors.
DataStructs.TanimotoSimilarity(FPs[10], FPs[10])

1.0

In [22]:
#Calculate the diagonal elements
for i in range(len(FPs)):
    for j in range(i, len(FPs)):  # Avoid duplicate calculations
        similarity = DataStructs.TanimotoSimilarity(FPs[i], FPs[j])
        similarity_matrix[i, j] = similarity
        similarity_matrix[j, i] = similarity

In [23]:
similarity_matrix.shape

(2957, 2957)

In [24]:
import seaborn as sns

In [25]:
similarity_matrix.min(), similarity_matrix.max(), similarity_matrix.mean()

(0.0, 1.0, 0.10686542950729885)

In [26]:
#Step 1: Compute Morgan fingerprints

In [27]:
# Suppose you have a list of SMILES strings for your molecules
def morgan_fp(smiles, radius=2, nBits=1024):
    mol = Chem.MolFromSmiles(smiles)
    fpgen = AllChem.GetMorganGenerator(fpSize=1024)
    arr_fp = fpgen.GetFingerprintAsNumPy(mol)
    return arr_fp

node_features = np.array([morgan_fp(s) for s in smiles_list])  # shape (1126, 1024)
y_label = np.array(y_label)
num_nodes = node_features.shape[0]

In [28]:
node_features.shape

(2957, 1024)

In [29]:
import numpy as np
import torch
import networkx as nx
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from torch_geometric.data import Data
from torch_geometric.utils import subgraph

In [30]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [31]:
y_label

array([3, 3, 3, ..., 0, 1, 1])

In [32]:
y

array(['II', 'II', 'II', ..., 'AA', 'AI', 'AI'], dtype=object)

In [33]:
df1.index.to_numpy()

array([   0,    1,    2, ..., 2954, 2955, 2956])

In [34]:
# Scaffold split (Bemis-Murcko), replacing random/stratified split to prevent
# structural leakage between train/val/test.
#
# Method: molecules are grouped by their Bemis-Murcko scaffold (Bemis & Murcko,
# J. Med. Chem. 1996); each scaffold group is then assigned WHOLE to a single
# split, so no two structurally near-identical molecules (sharing a scaffold)
# can appear on both sides of a train/test boundary. Scaffold groups are sorted
# largest-first and greedily filled into train, then val, then test until each
# target size is reached -- the standard convention used by MoleculeNet/DeepChem's
# ScaffoldSplitter.
#
# NOTE: unlike stratified random splitting, this does NOT guarantee proportional
# class representation in each split -- an entire rare class's molecules can
# share one scaffold and be assigned together to a single split. Given the small
# size of class IA (n=10) in this dataset, the diagnostics below explicitly check
# for this and must be reviewed before proceeding.

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict


def generate_scaffold(smiles, include_chirality=False):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=include_chirality)


def scaffold_split_indices(indices, smiles_list, y_label, train_ratio=0.6, val_ratio=0.2, seed=42):
    """
    Bemis-Murcko scaffold split. indices, smiles_list, and y_label must be
    positionally aligned (same order), matching the convention of the original
    stratified_split_indices function.

    Returns train_idx, val_idx, test_idx as arrays of the original index values.
    """
    indices = np.asarray(indices)
    smiles_arr = np.asarray(smiles_list)
    y_arr = np.asarray(y_label)
    assert len(indices) == len(smiles_arr) == len(y_arr), "indices, smiles_list, y_label must be the same length and aligned"

    scaffold_to_indices = defaultdict(list)
    n_invalid = 0
    for idx, smi in zip(indices, smiles_arr):
        scaffold = generate_scaffold(smi)
        if scaffold is None:
            n_invalid += 1
            scaffold = f"__invalid_{idx}__"   # unparseable SMILES: isolate as its own group
        elif scaffold == "":
            scaffold = f"__acyclic_{idx}__"    # fully acyclic molecule: no ring scaffold, isolate as its own group
        scaffold_to_indices[scaffold].append(idx)
    if n_invalid:
        print(f"WARNING: {n_invalid} molecule(s) had unparseable SMILES; each was isolated as its own scaffold group.")

    # Shuffle (for reproducibility/tie-breaking) then sort scaffold groups largest-first
    rng = np.random.default_rng(seed)
    scaffold_groups = list(scaffold_to_indices.values())
    order = rng.permutation(len(scaffold_groups))
    scaffold_groups = [scaffold_groups[i] for i in order]
    scaffold_groups.sort(key=len, reverse=True)

    n_total = len(indices)
    n_train_target = int(np.floor(train_ratio * n_total))
    n_val_target = int(np.floor(val_ratio * n_total))

    train_idx, val_idx, test_idx = [], [], []
    for group in scaffold_groups:
        if len(train_idx) + len(group) <= n_train_target:
            train_idx.extend(group)
        elif len(val_idx) + len(group) <= n_val_target:
            val_idx.extend(group)
        else:
            test_idx.extend(group)

    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


train_idx, val_idx, test_idx = scaffold_split_indices(
    df1.index.to_numpy(),
    df1['smiles'].to_numpy(),
    y_label,
    train_ratio=0.6,
    val_ratio=0.2,
    seed=42
)

print("Train size:", len(train_idx))
print("Val size:", len(val_idx))
print("Test size:", len(test_idx))

# --- Diagnostics: class distribution per split, and explicit leakage/coverage checks ---
y_series = pd.Series(np.asarray(y_label), index=df1.index.to_numpy())
class_names = {0: "AA", 1: "AI", 2: "IA", 3: "II"}  # adjust if your label encoding differs

print("\nClass distribution per split:")
dist_table = pd.DataFrame({
    "Train": y_series.loc[train_idx].value_counts(),
    "Val":   y_series.loc[val_idx].value_counts(),
    "Test":  y_series.loc[test_idx].value_counts(),
}).fillna(0).astype(int).sort_index()
dist_table.index = dist_table.index.map(lambda c: class_names.get(c, c))
print(dist_table)

# Correctness checks: no leakage, full coverage
all_idx = set(train_idx) | set(val_idx) | set(test_idx)
assert all_idx == set(df1.index.to_numpy()), "Index coverage mismatch after scaffold split!"
assert len(set(train_idx) & set(val_idx)) == 0, "Leakage detected: train/val overlap!"
assert len(set(train_idx) & set(test_idx)) == 0, "Leakage detected: train/test overlap!"
assert len(set(val_idx) & set(test_idx)) == 0, "Leakage detected: val/test overlap!"
print("\nNo scaffold leakage between splits; full coverage confirmed.")

# Explicit warning for any class missing (0 examples) from a split -- this WILL
# happen with small classes under scaffold splitting and needs your attention,
# not silent failure downstream (e.g. in classification_report or training).
for split_name, idx in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
    present_classes = set(y_series.loc[idx].unique())
    missing = set(class_names) - present_classes
    if missing:
        missing_names = [class_names[c] for c in missing]
        print(f"\n*** WARNING: {split_name} split has ZERO examples of class(es): {missing_names} ***")
        print("    This is a known risk of scaffold splitting with small classes (e.g. IA, n=10).")
        print("    Review before proceeding -- consider a different seed, a coarser scaffold")
        print("    (e.g. generic/skeleton scaffold), or explicitly discussing this limitation.")

Train size: 1774
Val size: 591
Test size: 592

Class distribution per split:
    Train  Val  Test
AA    193   41    41
AI    104   37    45
IA    564  146   105
II    913  367   401

No scaffold leakage between splits; full coverage confirmed.


In [35]:
svd = TruncatedSVD(n_components=256, random_state=42)
X = svd.fit_transform(node_features)

In [36]:
X.shape

(2957, 256)

In [37]:
#Mutual k-NN graph (with normalized squared weights)
k = 15
max_w = np.max(similarity_matrix ** 2)

edges = set()
for i in range(num_nodes):
    sims = similarity_matrix[i]
    nn_idx = np.argsort(sims)[::-1][1:k+1]
    for j in nn_idx:
        edges.add((i, j))

G = nx.Graph()
#G.add_nodes_from(range(num_nodes))  # keep isolated nodes

for i, j in edges:
    if (j, i) in edges:
        w = (similarity_matrix[i, j] ** 2) / max_w
        G.add_edge(i, j, weight=w)


In [38]:
G.number_of_nodes(), G.number_of_edges()

(2935, 15345)

In [39]:
#Convert to PyG Data object
edge_index = []
edge_attr = []

for u, v, data in G.edges(data=True):
    edge_index.append([u, v])
    edge_index.append([v, u])
    edge_attr.append([data["weight"]])
    edge_attr.append([data["weight"]])

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr, dtype=torch.float)

x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(y_label, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
data.num_classes = len(np.unique(y_label))


In [40]:
data

Data(x=[2957, 256], edge_index=[2, 30690], edge_attr=[30690, 1], y=[2957], num_classes=4)

In [41]:
#Create inductive subgraphs
#Training graph (only training nodes)
from torch_geometric.utils import subgraph

device = data.edge_index.device
train_idx = torch.as_tensor(train_idx).to(device)


train_edge_index, train_edge_weight = subgraph(
    train_idx,
    data.edge_index,
    data.edge_attr,
    relabel_nodes=True,
    num_nodes=data.num_nodes
)

train_data = Data(
    x=data.x[train_idx],
    y=data.y[train_idx],
    edge_index=train_edge_index,
    edge_weight=train_edge_weight
)

In [42]:
train_data

Data(x=[1774, 256], edge_index=[2, 17098], y=[1774], edge_weight=[17098, 1])

In [43]:
data.x[train_idx].shape, data.x[val_idx].shape, data.x[test_idx].shape

(torch.Size([1774, 256]), torch.Size([591, 256]), torch.Size([592, 256]))

In [44]:
#Validation graph (train + val nodes)

#Validation is still inductive relative to test nodes:
val_idx = torch.as_tensor(val_idx).to(device)
train_idx = torch.as_tensor(train_idx).to(device)

train_val_idx = torch.cat([train_idx, val_idx])

val_edge_index, val_edge_weight = subgraph(
    train_val_idx,
    data.edge_index,
    data.edge_attr,
    relabel_nodes=True,
    num_nodes=data.num_nodes
)

val_data = Data(
    x=data.x[train_val_idx],
    y=data.y[train_val_idx],
    edge_index=val_edge_index,
    edge_weight=val_edge_weight
)

# Mask for validation nodes (important!)
val_mask = torch.zeros(train_val_idx.size(0), dtype=torch.bool)
val_mask[torch.isin(train_val_idx, val_idx)] = True
val_data.val_mask = val_mask

In [45]:
#Test graph (train + val + test nodes)
test_edge_index, test_edge_weight = data.edge_index, data.edge_attr

test_data = Data(
    x=data.x,
    y=data.y,
    edge_index=test_edge_index,
    edge_weight=test_edge_weight
)

test_data.test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_data.test_mask[test_idx] = True

In [46]:
train_data

Data(x=[1774, 256], edge_index=[2, 17098], y=[1774], edge_weight=[17098, 1])

In [47]:
val_data

Data(x=[2365, 256], edge_index=[2, 23232], y=[2365], edge_weight=[23232, 1], val_mask=[2365])

In [48]:
test_data

Data(x=[2957, 256], edge_index=[2, 30690], y=[2957], edge_weight=[30690, 1], test_mask=[2957])

In [49]:
data

Data(x=[2957, 256], edge_index=[2, 30690], edge_attr=[30690, 1], y=[2957], num_classes=4)

In [50]:
G.number_of_nodes()

2935

In [51]:
data.edge_attr.shape

torch.Size([30690, 1])

In [52]:
data.x.shape

torch.Size([2957, 256])

In [53]:
data.has_self_loops(), data.has_isolated_nodes()

(True, True)

In [54]:
data.x[train_idx].shape

torch.Size([1774, 256])

In [55]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_curve, RocCurveDisplay, auc, roc_auc_score, matthews_corrcoef

In [119]:
#This config has been run on the 256-dimension normalized data
model = RandomForestClassifier(random_state=42, class_weight='balanced')
params = {'criterion': ['gini', 'entropy'],  'max_depth': list(range(3, 20, 2)), 'max_features': ['sqrt', 'log2', None], 'bootstrap': [True, False]}
grid = GridSearchCV(model, params, cv=5, n_jobs=-1, scoring='f1_macro')
grid.fit(data.x[train_idx], data.y[train_idx])

GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,
                                       15, 16, 17, 18, 19],
                         'max_features': ['sqrt', 'log2', None]})

In [56]:
#With scaffold_splitting:
model_rf = RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=5, max_features='log2', random_state=42)

In [57]:
#With 256 dimension
model_rf.fit(data.x[train_idx], data.y[train_idx])

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=5, max_features='log2', random_state=42)

In [58]:
#Test set
py_test = model_rf.predict(test_data.x[test_data.test_mask].cpu())
print(classification_report(test_data.y[test_data.test_mask].cpu(), py_test))

              precision    recall  f1-score   support

           0       0.38      0.20      0.26        41
           1       0.50      0.04      0.08        45
           2       0.32      0.33      0.33       105
           3       0.73      0.84      0.78       401

    accuracy                           0.65       592
   macro avg       0.48      0.35      0.36       592
weighted avg       0.62      0.65      0.61       592



In [59]:
matthews_corrcoef(test_data.y[test_data.test_mask], py_test)

0.19876938019711543

In [60]:
#Test set
py_test = model_rf.predict(data.x[test_idx].cpu())
print(classification_report(data.y[test_idx].cpu(), py_test))

              precision    recall  f1-score   support

           0       0.38      0.20      0.26        41
           1       0.50      0.04      0.08        45
           2       0.32      0.33      0.33       105
           3       0.73      0.84      0.78       401

    accuracy                           0.65       592
   macro avg       0.48      0.35      0.36       592
weighted avg       0.62      0.65      0.61       592



In [61]:
#With 256 dimension
roc_auc_score(data.y[test_idx], model_rf.predict_proba(data.x[test_idx]), multi_class='ovr', average='macro')

0.6835734618750127

In [62]:
#With 256 dimension
roc_auc_score(data.y[test_idx], model_rf.predict_proba(data.x[test_idx]), multi_class='ovr', average=None)

array([0.66714621, 0.73841154, 0.65940158, 0.66933452])

In [63]:
py_test = model_rf.predict(test_data.x[test_data.test_mask].cpu())
confusion_matrix(data.y[test_data.test_mask], py_test)

array([[  8,   1,  11,  21],
       [  2,   2,   4,  37],
       [  6,   0,  35,  64],
       [  5,   1,  58, 337]])

In [64]:
y_true = test_data.y[test_data.test_mask]
y_pred_rf = model_rf.predict(test_data.x[test_data.test_mask].cpu())

In [65]:
#Node classification task

In [66]:
data.edge_attr.shape

torch.Size([30690, 1])

In [67]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [68]:
#Similarity values are used in the train() function as edge_weight=data.edge_attr

In [69]:
import time

In [70]:
import torch
from torch import nn
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import scatter

In [76]:
class WeightedSAGEConv(MessagePassing):
    def __init__(self, in_channels, out_channels, bias=True):
        super().__init__(aggr='add')  # aggregation method: weighted sum
        self.lin = nn.Linear(in_channels, out_channels, bias=bias)

    def forward(self, x, edge_index, edge_weight):
        # x: [N, F]
        # edge_index: [2, E]
        # edge_weight: [E]
        return self.propagate(edge_index, x=x, edge_weight=edge_weight)

    def message(self, x_j, edge_weight):
        # x_j: neighbor features
        return edge_weight.view(-1, 1) * x_j

    def update(self, aggr_out):
        # Linear transformation after aggregation
        return self.lin(aggr_out)

In [72]:
#PyG’s SAGEConv uses mean by default
#If you want to use weighted-mean GraphSAGE aggregation:

In [80]:
from torch_geometric.utils import scatter

class WeightedMeanSAGEConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index, edge_weight=None):
        if edge_weight is None:
            edge_weight = x.new_ones(edge_index.size(1))

        self.edge_weight = edge_weight
        return self.propagate(edge_index, x=x, size=(x.size(0), x.size(0)))

    def message(self, x_j):
        return x_j * self.edge_weight.view(-1, 1)

    def aggregate(self, inputs, index, ptr=None, dim_size=None):
        weighted_sum = scatter(inputs, index, dim=0, dim_size=dim_size, reduce='sum')
        weight_sum = scatter(
            self.edge_weight, index, dim=0, dim_size=dim_size, reduce='sum'
        ).view(-1, 1)

        return weighted_sum / (weight_sum + 1e-8)

    def update(self, aggr_out):
        return self.lin(aggr_out)


In [81]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout
from torch_geometric.nn import BatchNorm

class GraphSAGE(torch.nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        #torch.manual_seed(1234)

        self.conv1 = WeightedMeanSAGEConv(num_features, 64)
        self.bn1 = BatchNorm(64)

        self.conv2 = WeightedMeanSAGEConv(64, 32)
        self.bn2 = BatchNorm(32)

        self.conv3 = WeightedMeanSAGEConv(32, 16)

        # Skip connections
        self.skip1 = Linear(num_features, 64) if num_features != 64 else None
        self.skip2 = Linear(64, 32)
        self.skip3 = Linear(32, 16)

        self.classifier = Linear(16, num_classes)
        self.dropout = Dropout(p=0.4)

    def forward(self, x, edge_index, edge_weight):
        # ---- Layer 1 ----
        residual = self.skip1(x) if self.skip1 is not None else x
        h = self.conv1(x, edge_index, edge_weight)
        h = F.relu(h)
        h = self.bn1(h)
        h = h + residual
        h = self.dropout(h)

        # ---- Layer 2 ----
        residual = self.skip2(h)
        h = self.conv2(h, edge_index, edge_weight)
        h = F.relu(h)
        h = self.bn2(h)
        h = h + residual
        h = self.dropout(h)

        # ---- Layer 3 ----
        residual = self.skip3(h)
        h = self.conv3(h, edge_index, edge_weight)
        h = h + residual

        out = self.classifier(h)
        return F.log_softmax(out, dim=-1), h


In [82]:
# =========================================================
# Inductive GraphSAGE training / validation / test (ONE CELL)
# =========================================================

import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score

# -------- Setup --------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

model = GraphSAGE(
    num_features=train_data.x.size(1),
    num_classes=int(train_data.y.max()) + 1
).to(device)

class_counts = torch.bincount(train_data.y)
class_weights = (1.0 / class_counts.float())
class_weights = class_weights / class_weights.sum() * len(class_counts)  # normalize
class_weights = class_weights.to(device)
print("Class weights:", class_weights.tolist())

criterion = torch.nn.NLLLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.01, weight_decay=5e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

# -------- Train --------
def train(data):
    model.train()
    optimizer.zero_grad()

    out, h = model(data.x, data.edge_index, data.edge_weight)
    loss = criterion(out, data.y)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    pred = out.argmax(dim=1)
    acc = (pred == data.y).float().mean()

    return loss, h, acc, out


# -------- Evaluate (Val / Test) --------
@torch.no_grad()
def evaluate(data, mask):
    model.eval()
    with torch.no_grad():
        out, h = model(data.x, data.edge_index, data.edge_weight)
        loss = criterion(out[mask], data.y[mask])
        pred = out[mask].argmax(dim=1)
        acc = (pred == data.y[mask]).float().mean()
        f1_macro = f1_score(data.y[mask].cpu(), pred.cpu(), average='macro', zero_division=0)
    return loss, acc, f1_macro, h, out


# -------- Training loop with early stopping --------
best_val_f1 = 0.0
patience, trials = 50, 0

print(
    "GraphSAGE (Inductive) | Train/Val/Test nodes:",
    f"{train_data.num_nodes}/"
    f"{val_data.val_mask.sum().item()}/"
    f"{test_data.test_mask.sum().item()}"
)

for epoch in range(1, 201):
    loss_train, h, acc_train, out = train(train_data)
    loss_val, acc_val, f1_val, _, _ = evaluate(val_data, val_data.val_mask)

    # Early stopping + checkpoint
    if f1_val > best_val_f1:     # was: acc_val > best_val_acc
        best_val_f1 = f1_val
        trials = 0
        torch.save(model.state_dict(), "best_graphSAGE_inductive.pt")
    else:
        trials += 1

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {loss_train:.4f} | Train Acc: {acc_train:.4f} | "
            f"Val Loss: {loss_val:.4f} | Val Acc: {acc_val:.4f} | "
            f"Best: {best_val_f1:.4f}"
        )

    if trials >= patience:
        print(f"Early stopping at epoch {epoch}")
        break


# -------- Test --------
checkpoint = torch.load("best_graphSAGE_inductive.pt", map_location=device)
model.load_state_dict(checkpoint)

loss_test, acc_test, f1, h_test, out_test = evaluate(
    test_data, test_data.test_mask
)

print(f"GraphSAGE Test Loss: {loss_test:.4f} | Test Acc: {acc_test:.4f}")


Class weights: [1.173241138458252, 2.177264928817749, 0.4014814496040344, 0.24801264703273773]
GraphSAGE (Inductive) | Train/Val/Test nodes: 1774/591/592
Epoch 010 | Train Loss: 0.5809 | Train Acc: 0.7120 | Val Loss: 1.3210 | Val Acc: 0.7039 | Best: 0.4922
Epoch 020 | Train Loss: 0.4883 | Train Acc: 0.7576 | Val Loss: 1.4874 | Val Acc: 0.6802 | Best: 0.5079
Epoch 030 | Train Loss: 0.4362 | Train Acc: 0.7824 | Val Loss: 1.8794 | Val Acc: 0.6565 | Best: 0.5097
Epoch 040 | Train Loss: 0.3739 | Train Acc: 0.8050 | Val Loss: 2.2747 | Val Acc: 0.6633 | Best: 0.5128
Epoch 050 | Train Loss: 0.3433 | Train Acc: 0.8247 | Val Loss: 2.6749 | Val Acc: 0.6599 | Best: 0.5128
Epoch 060 | Train Loss: 0.3045 | Train Acc: 0.8399 | Val Loss: 2.9877 | Val Acc: 0.6514 | Best: 0.5128
Epoch 070 | Train Loss: 0.2828 | Train Acc: 0.8467 | Val Loss: 3.1755 | Val Acc: 0.6514 | Best: 0.5128
Epoch 080 | Train Loss: 0.2600 | Train Acc: 0.8585 | Val Loss: 3.3797 | Val Acc: 0.6497 | Best: 0.5128
Early stopping at epoc

/tmp/ipykernel_108135/1286817838.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("best_graphSAGE_inductive.pt", map_location=device)


In [83]:
from sklearn.metrics import classification_report

# Ground truth and predictions for test nodes
y_true = test_data.y[test_data.test_mask].cpu()
y_pred = out_test[test_data.test_mask].argmax(dim=1).cpu()

# Print classification report
print(classification_report(y_true, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.438     0.341     0.384        41
           1      0.457     0.467     0.462        45
           2      0.320     0.381     0.348       105
           3      0.805     0.781     0.792       401

    accuracy                          0.655       592
   macro avg      0.505     0.492     0.496       592
weighted avg      0.667     0.655     0.660       592



In [84]:
matthews_corrcoef(y_true, y_pred)

0.3220101894798058

In [85]:
y_pred_gsage = y_pred

In [86]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize

def gnn_auc_scores(out, y, mask, n_classes=4):
    """out: log-softmax output [N, C] for ALL nodes (what model(...) returns first);
    y: labels [N]; mask: boolean test mask [N].
    Returns macro one-vs-rest AUROC, per-class AUROC, macro AUPRC, per-class AUPRC."""
    probs = out[mask].exp().detach().cpu().numpy().astype("float64")
    probs = probs / probs.sum(axis=1, keepdims=True)      # guard against tiny float32 drift
    y_np = y[mask].cpu().numpy()
    auroc_macro = roc_auc_score(y_np, probs, multi_class="ovr", average="macro")
    auroc_pc = roc_auc_score(y_np, probs, multi_class="ovr", average=None)
    y_bin = label_binarize(y_np, classes=list(range(n_classes)))
    auprc_pc = np.array([average_precision_score(y_bin[:, c], probs[:, c]) for c in range(n_classes)])
    return auroc_macro, auroc_pc, auprc_pc.mean(), auprc_pc

In [87]:
# out_test is the full-graph output your evaluate() already returns
auroc_macro, auroc_pc, auprc_macro, auprc_pc = gnn_auc_scores(out_test, test_data.y, test_data.test_mask)
print(f"Macro AUROC (OvR): {auroc_macro:.4f} | Macro AUPRC: {auprc_macro:.4f}")
for c, name in enumerate(["AA", "AI", "IA", "II"]):
    print(f"  {name}: AUROC={auroc_pc[c]:.4f}  AUPRC={auprc_pc[c]:.4f}")

Macro AUROC (OvR): 0.7570 | Macro AUPRC: 0.4895
  AA: AUROC=0.7490  AUPRC=0.3367
  AI: AUROC=0.7820  AUPRC=0.3781
  IA: AUROC=0.7269  AUPRC=0.3706
  II: AUROC=0.7700  AUPRC=0.8726


## Multi-run evaluation (10 seeds, fixed) — added to correct for run-to-run variability from unfixed GNN initialization.

Each run uses `set_seed(seed)` for `seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` (python/numpy/torch/cuda RNGs), trains GraphSAGE from scratch with early stopping (same protocol as the single-run cells above), and evaluates on the same fixed 226-node test set. Results are aggregated as mean +/- std across the 10 runs.

In [88]:
# ==========================================================================
# Multi-run aggregation helper (mean +/- std across seeds)
# ==========================================================================
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, matthews_corrcoef

CLASS_NAMES = ["AA", "AI", "IA", "II"]  # label order: 0=AA, 1=AI, 2=IA, 3=II

def summarize_runs(all_y_true, all_y_pred, all_acc, all_mcc, model_name, out_prefix):
    """
    all_y_true / all_y_pred: lists of numpy arrays, one pair per run (same
        test set each time, so all_y_true[i] should be identical across runs
        -- kept per-run only for safety/verification).
    all_acc / all_mcc: lists of floats, one per run.
    Returns (overall_df, per_class_df) and writes them to CSV.
    """
    n_runs = len(all_acc)

    reports = [
        classification_report(
            yt, yp, labels=list(range(len(CLASS_NAMES))),
            target_names=CLASS_NAMES, output_dict=True, zero_division=0
        )
        for yt, yp in zip(all_y_true, all_y_pred)
    ]
    # NEW: macro-F1 per run, pulled directly from each run's classification_report
    # -- directly tracks minority-class performance across seeds, the whole point
    # of the class-weighting + F1-based model-selection fix.
    all_f1_macro = [r["macro avg"]["f1-score"] for r in reports]

    overall_df = pd.DataFrame({
        "Metric": ["Accuracy", "MCC", "Macro_F1"],
        "Mean": [np.mean(all_acc), np.mean(all_mcc), np.mean(all_f1_macro)],
        "Std":  [np.std(all_acc),  np.std(all_mcc),  np.std(all_f1_macro)],
        "N_runs": [n_runs, n_runs, n_runs],
    })

    rows = []
    for cname in CLASS_NAMES:
        precisions = [r[cname]["precision"] for r in reports]
        recalls    = [r[cname]["recall"]    for r in reports]
        f1s        = [r[cname]["f1-score"]  for r in reports]
        rows.append({
            "Class": cname,
            "Precision_mean": np.mean(precisions), "Precision_std": np.std(precisions),
            "Recall_mean":    np.mean(recalls),    "Recall_std":    np.std(recalls),
            "F1_mean":        np.mean(f1s),        "F1_std":        np.std(f1s),
        })
    per_class_df = pd.DataFrame(rows)

    print(f"\n=== {model_name}: overall performance across {n_runs} runs (mean +/- std) ===")
    print(overall_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print(f"\n=== {model_name}: per-class performance across {n_runs} runs (mean +/- std) ===")
    print(per_class_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    overall_df.to_csv(f"{out_prefix}_overall_multirun.csv", index=False)
    per_class_df.to_csv(f"{out_prefix}_per_class_multirun.csv", index=False)
    print(f"\nSaved: {out_prefix}_overall_multirun.csv, {out_prefix}_per_class_multirun.csv")

    return overall_df, per_class_df

In [89]:
# ==========================================================================
# GraphSAGE: fixed-seed, 10-run training for mean +/- std reporting
# (accuracy, MCC, per-class P/R/F1 via summarize_runs; plus AUROC / AUPRC)
# ==========================================================================
import numpy as np
import pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, roc_auc_score,
                             average_precision_score)
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ["AA", "AI", "IA", "II"]

def gnn_auc_scores(out, y, mask, n_classes=4):
    """out: log-softmax output [N, C] for ALL nodes (what model(...) returns first);
    y: labels [N]; mask: boolean test mask [N].
    Returns macro one-vs-rest AUROC, per-class AUROC, macro AUPRC, per-class AUPRC."""
    probs = out[mask].exp().detach().cpu().numpy().astype("float64")
    probs = probs / probs.sum(axis=1, keepdims=True)      # guard against tiny float32 drift
    y_np = y[mask].cpu().numpy()
    auroc_macro = roc_auc_score(y_np, probs, multi_class="ovr", average="macro")
    auroc_pc = roc_auc_score(y_np, probs, multi_class="ovr", average=None)
    y_bin = label_binarize(y_np, classes=list(range(n_classes)))
    auprc_pc = np.array([average_precision_score(y_bin[:, c], probs[:, c]) for c in range(n_classes)])
    return auroc_macro, auroc_pc, auprc_pc.mean(), auprc_pc

SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# class_weights computed ONCE outside the loop -- train_data.y is identical
# across all 10 seeds (only the model's random initialization differs)
class_counts = torch.bincount(train_data.y)
class_weights = (1.0 / class_counts.float())
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = class_weights.to(device)
print("Class weights:", class_weights.tolist())
criterion = torch.nn.NLLLoss(weight=class_weights)

all_y_true, all_y_pred, all_acc, all_mcc = [], [], [], []
all_auroc_macro, all_auroc_pc, all_auprc_macro, all_auprc_pc = [], [], [], []

for run_idx, seed in enumerate(SEEDS):
    set_seed(seed)  # resets python/numpy/torch(+cuda) RNG state for this run

    model = GraphSAGE(num_features=train_data.x.size(1),
                             num_classes=int(train_data.y.max()) + 1).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

    def train_step(data):
        model.train()
        optimizer.zero_grad()
        out, h = model(data.x, data.edge_index, data.edge_weight)
        loss = criterion(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pred = out.argmax(dim=1)
        acc = (pred == data.y).float().mean()
        return loss, acc

    @torch.no_grad()
    def eval_step(data, mask):
        model.eval()
        out, h = model(data.x, data.edge_index, data.edge_weight)
        loss = criterion(out[mask], data.y[mask])
        pred = out[mask].argmax(dim=1)
        acc = (pred == data.y[mask]).float().mean()
        f1_macro = f1_score(data.y[mask].cpu(), pred.cpu(), average="macro", zero_division=0)
        return loss, acc, f1_macro, out

    best_val_f1 = 0.0
    patience, trials = 50, 0
    best_state = None
    ckpt_path = f"best_graphSAGE_inductive_multirun_seed{seed}.pt"

    for epoch in range(1, 201):
        loss_train, acc_train = train_step(train_data)
        loss_val, acc_val, f1_val, _ = eval_step(val_data, val_data.val_mask)

        if f1_val > best_val_f1:
            best_val_f1 = f1_val
            trials = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            trials += 1
        if trials >= patience:
            break

    # ---- final test evaluation with the best (validation macro-F1) checkpoint ----
    model.load_state_dict(best_state)
    loss_test, acc_test, f1_test, out_test = eval_step(test_data, test_data.test_mask)

    # AUROC / AUPRC: must come AFTER out_test has been computed for THIS seed
    am, apc, pm, ppc = gnn_auc_scores(out_test, test_data.y, test_data.test_mask)
    all_auroc_macro.append(am); all_auroc_pc.append(apc)
    all_auprc_macro.append(pm); all_auprc_pc.append(ppc)

    y_true_run = test_data.y[test_data.test_mask].cpu().numpy()
    y_pred_run = out_test[test_data.test_mask].argmax(dim=1).cpu().numpy()
    mcc_run = matthews_corrcoef(y_true_run, y_pred_run)

    all_y_true.append(y_true_run)
    all_y_pred.append(y_pred_run)
    all_acc.append(acc_test.item())
    all_mcc.append(mcc_run)

    print(f"[GraphSAGE] Run {run_idx+1}/{len(SEEDS)} (seed={seed}): "
          f"Test Acc={acc_test.item():.4f}, Test F1(macro)={f1_test:.4f}, MCC={mcc_run:.4f}, "
          f"AUROC(macro)={am:.4f}, best epoch val_f1={best_val_f1:.4f}")

# ---- aggregate AUROC / AUPRC across seeds (population SD, same convention as the other multirun CSVs) ----
A, P = np.array(all_auroc_pc), np.array(all_auprc_pc)
rows = [{"Class": "macro",
         "AUROC_mean": np.mean(all_auroc_macro), "AUROC_std": np.std(all_auroc_macro),
         "AUPRC_mean": np.mean(all_auprc_macro), "AUPRC_std": np.std(all_auprc_macro)}]
for c, name in enumerate(CLASS_NAMES):
    rows.append({"Class": name,
                 "AUROC_mean": A[:, c].mean(), "AUROC_std": A[:, c].std(),
                 "AUPRC_mean": P[:, c].mean(), "AUPRC_std": P[:, c].std()})
auc_df = pd.DataFrame(rows)
auc_df.to_csv("gsage_auc_multirun.csv", index=False)   # change the prefix per architecture
print("\n=== GraphSAGE: AUROC / AUPRC across", len(SEEDS), "runs (mean +/- std) ===")
print(auc_df.round(4).to_string(index=False))

overall_graphSAGE, per_class_graphSAGE = summarize_runs(
    all_y_true, all_y_pred, all_acc, all_mcc,
    model_name="GraphSAGE", out_prefix="gsage"
)

Class weights: [1.173241138458252, 2.177264928817749, 0.4014814496040344, 0.24801264703273773]
[GraphSAGE] Run 1/10 (seed=0): Test Acc=0.6605, Test F1(macro)=0.4768, MCC=0.3128, AUROC(macro)=0.7450, best epoch val_f1=0.5268
[GraphSAGE] Run 2/10 (seed=1): Test Acc=0.6841, Test F1(macro)=0.4865, MCC=0.3305, AUROC(macro)=0.7832, best epoch val_f1=0.5244
[GraphSAGE] Run 3/10 (seed=2): Test Acc=0.6791, Test F1(macro)=0.4835, MCC=0.3358, AUROC(macro)=0.7512, best epoch val_f1=0.5409
[GraphSAGE] Run 4/10 (seed=3): Test Acc=0.6689, Test F1(macro)=0.4576, MCC=0.2870, AUROC(macro)=0.7603, best epoch val_f1=0.5247
[GraphSAGE] Run 5/10 (seed=4): Test Acc=0.6436, Test F1(macro)=0.4572, MCC=0.2907, AUROC(macro)=0.7496, best epoch val_f1=0.5273
[GraphSAGE] Run 6/10 (seed=5): Test Acc=0.6807, Test F1(macro)=0.4707, MCC=0.3108, AUROC(macro)=0.7603, best epoch val_f1=0.5208
[GraphSAGE] Run 7/10 (seed=6): Test Acc=0.6622, Test F1(macro)=0.4752, MCC=0.3082, AUROC(macro)=0.7449, best epoch val_f1=0.5362
[G